# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd

In [2]:
df_clean = pd.read_csv("insurance_clean.csv")

In [3]:
df_clean.head()

,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,UNKNOWN,Master,588174,0,1000,0,Personal Auto,Four-Door Car,2
1,QZ44356,Arizona,F,Bachelor,697953,0,94,0,Personal Auto,Four-Door Car,1131
2,AI49188,Nevada,F,Bachelor,1288743,48767,108,0,Personal Auto,Two-Door Car,566
3,WW63253,California,M,Bachelor,764586,0,106,0,Corporate Auto,SUV,529
4,GA49547,Washington,M,High School or Below,536307,36357,68,0,Personal Auto,Four-Door Car,17


In [4]:
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

print(df2.shape, df3.shape)

(996, 11) (7070, 11)


In [5]:
df2.head()

,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Total Claim Amount,Policy Type,Vehicle Class
0,GS98873,Arizona,F,Bachelor,323912.47%,16061,88,1/0/00,633.6,Personal Auto,Four-Door Car
1,CW49887,California,F,Master,462680.11%,79487,114,1/0/00,547.2,Special Auto,SUV
2,MY31220,California,F,College,899704.02%,54230,112,1/0/00,537.6,Personal Auto,Two-Door Car
3,UH35128,Oregon,F,College,2580706.30%,71210,214,1/1/00,1027.2,Personal Auto,Luxury Car
4,WH52799,Arizona,F,College,380812.21%,94903,94,1/0/00,451.2,Corporate Auto,Two-Door Car


In [6]:
df3.head()

,Customer,State,Customer Lifetime Value,Education,Gender,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Total Claim Amount,Vehicle Class
0,SA25987,Washington,3479.137523,High School or Below,M,0,104,0,Personal Auto,499.200000,Two-Door Car
1,TB86706,Arizona,2502.637401,Master,M,0,66,0,Personal Auto,3.468912,Two-Door Car
2,ZL73902,Nevada,3265.156348,Bachelor,F,25820,82,0,Personal Auto,393.600000,Four-Door Car
3,KX23516,California,4455.843406,High School or Below,F,0,121,0,Personal Auto,699.615192,SUV
4,FN77294,California,7704.958480,High School or Below,M,30366,101,2,Personal Auto,484.800000,SUV


In [7]:
def clean_like_df_clean(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )

    
    if "st" in df.columns:
        df = df.rename(columns={"st": "state"})

    col_map = {
        "customer": "customer",
        "state": "state",
        "gender": "gender",
        "education": "education",
        "customer_lifetime_value": "customer_lifetime_value",
        "income": "income",
        "monthly_premium_auto": "monthly_premium_auto",
        "number_of_open_complaints": "number_of_open_complaints",
        "policy_type": "policy_type",
        "vehicle_class": "vehicle_class",
        "total_claim_amount": "total_claim_amount"
    }

   
    df = df[[c for c in df.columns if c in col_map]].rename(columns=col_map)


    df["gender"] = df["gender"].astype(str).str.upper().str.strip()
    gender_map = {
        "F": "F", "FEMALE": "F", "FEMAL": "F",
        "M": "M", "MALE": "M"
    }
    df["gender"] = df["gender"].replace(gender_map)


    df["state"] = df["state"].astype(str).replace({
        "AZ": "Arizona",
        "Cali": "California",
        "WA": "Washington"
    })


    df["education"] = df["education"].replace({"Bachelors": "Bachelor"})


    df["customer_lifetime_value"] = (
        df["customer_lifetime_value"].astype(str).str.replace("%", "", regex=False)
    )
    df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors="coerce")


    df["number_of_open_complaints"] = (
        df["number_of_open_complaints"].astype(str).str.split("/").str[1]
    )
    df["number_of_open_complaints"] = pd.to_numeric(df["number_of_open_complaints"], errors="coerce")


    df["vehicle_class"] = df["vehicle_class"].replace({
        "Sports Car": "Luxury",
        "Luxury SUV": "Luxury",
        "Luxury Car": "Luxury"
    })


    num_cols = [
        "customer_lifetime_value", "income",
        "monthly_premium_auto", "number_of_open_complaints",
        "total_claim_amount"
    ]
    cat_cols = ["state", "gender", "education", "policy_type", "vehicle_class"]

    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")


     
    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)
            df[col] = df[col].round().astype(int)

  
    df = df.drop_duplicates().reset_index(drop=True)

    return df

In [8]:
df2_clean = clean_like_df_clean(df2)
df3_clean = clean_like_df_clean(df3)

print(df2_clean.head())
print(df3_clean.head())
print(df2_clean.info())
print(df3_clean.info())

  customer       state gender education  customer_lifetime_value  income  \
0  GS98873     Arizona      F  Bachelor                   323912   16061   
1  CW49887  California      F    Master                   462680   79487   
2  MY31220  California      F   College                   899704   54230   
3  UH35128      Oregon      F   College                  2580706   71210   
4  WH52799     Arizona      F   College                   380812   94903   

   monthly_premium_auto  number_of_open_complaints  total_claim_amount  \
0                    88                          0                 634   
1                   114                          0                 547   
2                   112                          0                 538   
3                   214                          1                1027   
4                    94                          0                 451   

      policy_type  vehicle_class  
0   Personal Auto  Four-Door Car  
1    Special Auto           

In [9]:
df_all = pd.concat([df_clean, df2_clean, df3_clean], ignore_index=True)
print(df_all.shape)

(12074, 11)


In [10]:
df_all.head()

,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,UNKNOWN,Master,588174,0,1000,0,Personal Auto,Four-Door Car,2
1,QZ44356,Arizona,F,Bachelor,697953,0,94,0,Personal Auto,Four-Door Car,1131
2,AI49188,Nevada,F,Bachelor,1288743,48767,108,0,Personal Auto,Two-Door Car,566
3,WW63253,California,M,Bachelor,764586,0,106,0,Corporate Auto,SUV,529
4,GA49547,Washington,M,High School or Below,536307,36357,68,0,Personal Auto,Four-Door Car,17


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [11]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)

In [13]:
pivot_revenue = (
    df.pivot_table(
        index="sales_channel",
        values="total_claim_amount",
        aggfunc="sum"
    )
    .round(2)
    .sort_values("total_claim_amount", ascending=False)
)

print(pivot_revenue)


               total_claim_amount
sales_channel                    
Agent                  1810226.82
Branch                 1301204.00
Call Center             926600.82
Web                     706600.04


In [14]:
pivot_clv = (
    df.pivot_table(
        index="gender",
        columns="education",
        values="customer_lifetime_value",
        aggfunc="mean"
    )
    .round(2)
)

print(pivot_clv)

education  Bachelor  College   Doctor  High School or Below   Master
gender                                                              
F           7874.27  7748.82  7328.51               8675.22  8157.05
M           7703.60  8052.46  7415.33               8149.69  8168.83


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [15]:

df["effective_to_date"] = pd.to_datetime(df["effective_to_date"])
df["month"] = df["effective_to_date"].dt.month_name()   # 例如 "January", "February"


complaints_long = (
    df.groupby(["policy_type", "month"], as_index=False)
      ["number_of_open_complaints"]
      .sum()
      .rename(columns={"number_of_open_complaints": "total_complaints"})
)


complaints_long.head()

,policy_type,month,total_complaints
0,Corporate Auto,February,385.208135
1,Corporate Auto,January,443.434952
2,Personal Auto,February,1453.684441
3,Personal Auto,January,1727.605722
4,Special Auto,February,95.226817


In [16]:
complaints_long.sort_values("total_complaints", ascending=False).head(10)

,policy_type,month,total_complaints
3,Personal Auto,January,1727.605722
2,Personal Auto,February,1453.684441
1,Corporate Auto,January,443.434952
0,Corporate Auto,February,385.208135
4,Special Auto,February,95.226817
5,Special Auto,January,87.074049
